Week 1 - Polynomial and Interaction Terms, Multicollinearity, and VIF

This notebook applies Week 1 regression concepts to both Capstone datasets:

* Dataset A - Credit Card Transactions:
  55,000 transactions. Regression target = log(transaction amount)
* Dataset B - UNSW-NB15 network intrusion: 
  50,000-row stratified sample of network flows. Regression target = log(flow duration, `dur`)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
RNG = 42

cc = pd.read_csv("data/credit_card_fraud_dataset.csv")
cc["log_amount"] = np.log1p(cc["transaction_amount_usd"])
cc_continuous = ["credit_utilization_pct", "avg_monthly_spend_usd", "annual_income_usd",
                 "credit_limit_usd", "distance_from_home_km", "num_transactions_last_30d",
                 "age", "credit_score", "account_age_months", "velocity_last_1h"]
cc_binary = ["is_international_transaction", "is_new_merchant", "is_night_transaction",
             "card_present", "cvv_mismatch", "device_changed"]
cc_categorical = ["merchant_category", "device_type", "occupation"]
cc_clf_feats = ["cvv_mismatch", "prev_fraud_flags", "failed_attempts_last_24h", "device_changed",
                "is_international_transaction", "is_new_merchant", "card_present",
                "is_night_transaction", "velocity_last_1h", "credit_utilization_pct",
                "transaction_amount_usd", "distance_from_home_km", "num_transactions_last_30d",
                "age", "credit_score"]

def cc_regression_design():
    X = pd.concat([
        cc[cc_continuous].reset_index(drop=True),
        cc[cc_binary].reset_index(drop=True).astype(float),
        pd.get_dummies(cc[cc_categorical], drop_first=True).reset_index(drop=True).astype(float),
    ], axis=1)
    return X, cc["log_amount"].values

def cc_classification_design():
    return cc[cc_clf_feats].copy(), cc["is_fraud"].values

unsw_full = pd.read_parquet("data_unsw/UNSW_NB15_training-set.parquet")
unsw = unsw_full.groupby("label", group_keys=False).sample(frac=50000/len(unsw_full), random_state=RNG)
unsw = unsw.reset_index(drop=True)
unsw["log_dur"] = np.log1p(unsw["dur"].clip(lower=0))
_top_proto = unsw["proto"].value_counts().head(6).index
unsw["proto_grp"] = np.where(unsw["proto"].isin(_top_proto), unsw["proto"].astype(str), "other")
unsw_cats = ["service", "state", "proto_grp"]
unsw_numeric = [c for c in unsw.select_dtypes(include=[np.number]).columns
                if c not in ("label", "dur", "log_dur")]

def _unsw_dummies():
    return pd.get_dummies(unsw[unsw_cats].astype(str), drop_first=True).reset_index(drop=True).astype(float)

def unsw_regression_design():
    """UNSW regression: predict log(flow duration). Returns (X df, y)."""
    X = pd.concat([unsw[unsw_numeric].reset_index(drop=True), _unsw_dummies()], axis=1)
    return X, unsw["log_dur"].values

def unsw_classification_design():
    """UNSW classification: predict attack label (1=attack, 0=normal). Returns (X df, y)."""
    num = [c for c in unsw.select_dtypes(include=[np.number]).columns if c not in ("label", "log_dur")]
    X = pd.concat([unsw[num].reset_index(drop=True), _unsw_dummies()], axis=1)
    return X, unsw["label"].values

print("Credit card:", cc.shape, "| fraud rate", round(cc.is_fraud.mean(), 3))
print("UNSW sample:", unsw.shape, "| attack rate", round(unsw.label.mean(), 3))


Credit card: (55000, 30) | fraud rate 0.038
UNSW sample: (50000, 38) | attack rate 0.681


In [2]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

def evaluate(Xdf, y, label, results):
    Xtr, Xte, ytr, yte = train_test_split(Xdf.values, y, test_size=0.25, random_state=RNG)
    m = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)
    pred = m.predict(Xte)
    results[label] = {"test_R2": round(r2_score(yte, pred), 3),
                      "test_RMSE": round(root_mean_squared_error(yte, pred), 3),
                      "n_features": Xdf.shape[1]}
    print(f"{label:32s} R2={results[label]['test_R2']:.3f}  RMSE={results[label]['test_RMSE']:.3f}  p={Xdf.shape[1]}")
    return results

def add_poly(Xdf, subset, degree=2):
    pf = PolynomialFeatures(degree=degree, include_bias=False)
    arr = pf.fit_transform(Xdf[subset])
    names = pf.get_feature_names_out(subset)
    keep = [i for i, n in enumerate(names) if ("^2" in n) or (" " in n)]  # squares + interactions only
    extra = pd.DataFrame(arr[:, keep], columns=[names[i] for i in keep])
    return pd.concat([Xdf.reset_index(drop=True), extra.reset_index(drop=True)], axis=1)

def vif_table(df_sub):
    Z = StandardScaler().fit_transform(df_sub)
    v = [variance_inflation_factor(Z, i) for i in range(Z.shape[1])]
    return pd.DataFrame({"feature": df_sub.columns, "VIF": np.round(v, 2)}).sort_values("VIF", ascending=False).reset_index(drop=True)

## Dataset A - Credit card: predicting transaction amount

I first regress `log(amount)` on the continuous features only, then add the one-hot
categorical features, then add degree-2 polynomial and interaction terms built from the
four strongest continuous drivers.

In [3]:
ccX, ccy = cc_regression_design()
cc_res = {}
evaluate(ccX[cc_continuous], ccy, "A1. continuous only", cc_res)
evaluate(ccX, ccy, "A2. + categorical + flags", cc_res)
cc_poly_subset = ["credit_utilization_pct", "avg_monthly_spend_usd",
                  "distance_from_home_km", "num_transactions_last_30d"]
ccX_poly = add_poly(ccX, cc_poly_subset)
evaluate(ccX_poly, ccy, "A3. + polynomial/interaction", cc_res)
pd.DataFrame(cc_res).T

A1. continuous only              R2=0.659  RMSE=0.715  p=10
A2. + categorical + flags        R2=0.813  RMSE=0.529  p=44
A3. + polynomial/interaction     R2=0.875  RMSE=0.433  p=54


,test_R2,test_RMSE,n_features
A1. continuous only,0.659,0.715,10.0
A2. + categorical + flags,0.813,0.529,44.0
A3. + polynomial/interaction,0.875,0.433,54.0


The categorical features give the largest jump, and the polynomial/interaction terms add
a further improvement, confirming mild curvature in the spending relationships. Next I
check multicollinearity among the continuous features.

In [4]:
cc_vif = vif_table(cc[cc_continuous])
print(cc_vif.to_string(index=False))

                  feature  VIF
        annual_income_usd 3.53
         credit_limit_usd 2.41
    avg_monthly_spend_usd 2.26
    distance_from_home_km 1.35
   credit_utilization_pct 1.32
         velocity_last_1h 1.28
num_transactions_last_30d 1.13
                      age 1.00
             credit_score 1.00
       account_age_months 1.00


`annual_income_usd`, `credit_limit_usd`, and `avg_monthly_spend_usd` carry the highest
VIFs (about 2.3-3.5) versus roughly 1.0 for every other feature. A customer's income,
limit, and spending move together. The values sit below the strict threshold of 5, so the
collinearity here is only moderate. The model still predicts well but these three
coefficients should be interpreted with care. Week 2 dampens this redundancy with
regularization.

## Dataset B - UNSW: predicting flow duration

Flow duration is far more nonlinear than transaction amount so I expected polynomial and
interaction terms to help a lot here. I pick the polynomial subset data-drivenly as the
continuous features most correlated with `log(dur)`.

In [5]:
uX, uy = unsw_regression_design()
u_res = {}
evaluate(uX, uy, "B1. linear (all features)", u_res)

cont_candidates = [c for c in unsw_numeric if unsw[c].nunique() > 10]
corrs = pd.Series({c: np.corrcoef(unsw[c], uy)[0, 1] for c in cont_candidates}).abs().sort_values(ascending=False)
u_poly_subset = corrs.head(5).index.tolist()
print("Polynomial subset (top correlates of log_dur):", u_poly_subset)

uX_poly = add_poly(uX, u_poly_subset)
evaluate(uX_poly, uy, "B2. + polynomial/interaction", u_res)
pd.DataFrame(u_res).T

B1. linear (all features)        R2=0.649  RMSE=0.385  p=54
Polynomial subset (top correlates of log_dur): ['tcprtt', 'synack', 'ackdat', 'rate', 'dtcpb']
B2. + polynomial/interaction     R2=0.667  RMSE=0.376  p=69


,test_R2,test_RMSE,n_features
B1. linear (all features),0.649,0.385,54.0
B2. + polynomial/interaction,0.667,0.376,69.0


Contrary to my expectation, the polynomial and interaction terms add only a marginal
gain for UNSW.
Two things explain this. First, the one-hot `service`, `state`, and `proto` features already
let the linear model reach ~0.65. Second, I found that adding
degree-2 terms built from the heavy-tailed flow-rate features (`rate`, `sload`, `dload`)
actually hurts the model badly because squaring a long-tailed variable creates extreme
leverage points. Only the bounded handshake-timing features could be expanded safely and
they carry limited duration signal.

In [6]:
u_vif_subset = ["sbytes", "sloss", "spkts", "dbytes", "dloss", "dpkts",
                "swin", "dwin", "is_ftp_login", "ct_ftp_cmd", "rate", "sload"]
u_vif = vif_table(unsw[u_vif_subset])
print(u_vif.to_string(index=False))

     feature    VIF
is_ftp_login    inf
  ct_ftp_cmd    inf
       dloss 714.56
       dpkts 385.87
       sloss 383.29
      sbytes 317.59
      dbytes 228.07
       spkts  73.27
        dwin  48.48
        swin  48.21
        rate   1.90
       sload   1.58


UNSW's multicollinearity is far more extreme than the credit card data. `is_ftp_login`
and `ct_ftp_cmd` are essentially the same column and the
byte/packet/loss features (`sbytes`/`sloss`/`spkts`; `dbytes`/`dloss`/`dpkts`) are nearly
redundant because bytes, packets, and losses scale together in a network flow.

## Conclusions

* Polynomial and interaction terms help both datasets but unequally and not as I
  predicted. They add a solid gain for credit-card amount but only a marginal
  gain for UNSW duration.
* Both datasets are multicollinear. UNSW far more so. VIF made this concrete and motivates the regularization and dimension-reduction methods that follow.
